In [2]:
import numpy as np
import pandas as pd
import itertools
import matplotlib.pyplot as plt
from datetime import date
import re
from tqdm import tqdm
import time
from pybedtools import BedTool
import random
import math
import pickle
import seaborn as sns
from scipy.stats import percentileofscore, pearsonr
from scipy.spatial.distance import jaccard, euclidean, cosine
import functools
import operator
import gzip
from sklearn.feature_extraction.text import TfidfVectorizer

today = date.today()

**SNP**

**Transcription Factor**

**DHS**

**Chromatin State**

**Genes**

In [3]:
# 25kb bins
bin_cols = ['seqname', 'start', 'end', '?']
bins_25kb = pd.read_csv('../data/Altius_100k_epilogos_hg19_WM20210518.bed', delimiter='\t', names=bin_cols)
print(bins_25kb)


# SNP data
snp_cols = ['Chromosome', 'p1', 'p2', 'Reference SNP cluster ID', 'pvalue', 'phenotype']
snp_df = pd.read_csv('../data/UKBB_all_traits_all_sig_associations_CT20220207.bed', delimiter='\t', names=snp_cols)
print(snp_df)

# trait_codes = []
# with gzip.open('/home/nasi4/proj/GWAS/UKBB_201807/info/phenotypes.both_sexes.tsv.gz','r') as f:
#     for line in f:
#         trait_codes.append(line.decode("utf-8").strip('\n').split('\t'))

# trait_codes_df = pd.DataFrame(trait_codes[1:], columns=trait_codes[0])
# print(trait_codes_df)

# TF data 
# NOTE: THIS IS CURRENTLY HG38
tf_pd = pd.read_pickle('../data/TF-df-hg38.pkl')
print(tf_pd)

# DHS data
dhs_df = pd.read_pickle('../data/dhs_metadata.pkl')
print(dhs_df)

dhs_Mixture_df = pd.read_pickle('../data/dhs_mixture_df.pkl')
print(dhs_Mixture_df)

dhs_obs_over_exp = pd.read_pickle('../data/dhs_obs_over_exp_25kb.pkl')
print(dhs_obs_over_exp)

# Chromatin State Data
epilogos_833_df = pd.read_pickle('../data/epilogos_state_data_18.pkl')
print(epilogos_833_df)

epilogos_states_df = pd.read_pickle('../data/epilogos_states_df')
print(epilogos_states_df)

epilogos_metadata = pd.read_csv('../data/All_833_biosamples/metadata.tsv', delimiter='\t')
print(epilogos_metadata)


# Gene Data
string_assoc = pd.read_pickle('../data/hg19-protein-associations.pkl')
print(string_assoc)

gene_hg19_bed = pd.read_pickle('../data/hg19_gene_bed.pkl')
print(gene_hg19_bed)

protein_assoc_hg19 = pd.read_pickle('../data/hg19-protein-associations-detailed.pkl')
print(protein_assoc_hg19)

      seqname      start        end     ?
0        chr1       9000      34000  0.41
1        chr1      34000      59000  0.41
2        chr1      61000      86000  0.41
3        chr1      86000     111000  0.41
4        chr1     111000     136000  0.41
...       ...        ...        ...   ...
99991    chrX  155042800  155067800  0.41
99992    chrX  155067800  155092800  0.41
99993    chrX  155092800  155117800  0.41
99994    chrX  155117800  155142800  0.41
99995    chrX  155142800  155167800  0.41

[99996 rows x 4 columns]
         Chromosome         p1         p2 Reference SNP cluster ID  \
0             chr12  130300801  130300802               rs73150827   
1              chr4   99318162   99318163                rs1229984   
2              chr9    8303561    8303562              rs112120933   
3              chr1   46941503   46941504              rs141586057   
4             chr20   52932649   52932650              rs185961463   
...             ...        ...        ...         

         seqname      start        end   0   1   2   3   4   5   6  ...  823  \
0           chr1          0        200  18  18  18  18  18  18  18  ...   18   
1           chr1        200        400  18  18  18  18  18  18  18  ...   18   
2           chr1        400        600  18  18  18  18  18  18  18  ...   18   
3           chr1        600        800  18  18  18  18  18  18  18  ...   18   
4           chr1        800       1000  18  18  18  18  18  18  18  ...   18   
...          ...        ...        ...  ..  ..  ..  ..  ..  ..  ..  ...  ...   
15181503    chrX  155269400  155269600  18  18  18  18  18  18  18  ...   18   
15181504    chrX  155269600  155269800  18  18  18  18  18  18  18  ...   18   
15181505    chrX  155269800  155270000  18  18  18  18  18  18  18  ...   18   
15181506    chrX  155270000  155270200  18  18  18  18  18  18  18  ...   18   
15181507    chrX  155270200  155270400  18  18  18  18  18  18  18  ...   18   

          824  825  826  827  828  829 

                                  protein2  neighborhood  fusion  cooccurence  \
protein1                                                                        
9606.ENSP00000000233  9606.ENSP00000379496           0.0     0.0          0.0   
9606.ENSP00000000233  9606.ENSP00000314067           0.0     0.0          0.0   
9606.ENSP00000000233  9606.ENSP00000263116           0.0     0.0          0.0   
9606.ENSP00000000233  9606.ENSP00000361263           0.0     0.0          0.0   
9606.ENSP00000000233  9606.ENSP00000409666           0.0     0.0          0.0   
...                                    ...           ...     ...          ...   
9606.ENSP00000469413  9606.ENSP00000397459           0.0     0.0          0.0   
9606.ENSP00000469413  9606.ENSP00000222399           0.0     0.0          0.0   
9606.ENSP00000469413  9606.ENSP00000215659           0.0     0.0        144.0   
9606.ENSP00000469413  9606.ENSP00000305918           0.0     0.0          0.0   
9606.ENSP00000469413  9606.E

# 1kb

In [83]:
bins_1kb = pd.DataFrame()
for i in bins_25kb.index:
    temp_bins = pd.DataFrame()
    seqname = bins_25kb.loc[i]['seqname']
    start = bins_25kb.loc[i]['start']
    for j in range(25):
        index = i*25 +j
        temp_bins[index] = ([seqname, start, start+1000])
        start = start + 1000
    bins_1kb = pd.concat([bins_1kb, temp_bins.T], axis=0)
#     print(bins_1kb)
    if i % 100 == 0:
        print(i)
bins_1kb

0
100
200
300
400
500
600
700
800
900
1000
1100
1200
1300
1400
1500
1600
1700
1800
1900
2000
2100
2200
2300
2400
2500
2600
2700
2800
2900
3000
3100
3200
3300
3400
3500
3600
3700
3800
3900
4000
4100
4200
4300
4400
4500
4600
4700
4800
4900
5000
5100
5200
5300
5400
5500
5600
5700
5800
5900
6000
6100
6200
6300
6400
6500
6600
6700
6800
6900
7000
7100
7200
7300
7400
7500
7600
7700
7800
7900
8000
8100
8200
8300
8400
8500
8600
8700
8800
8900
9000
9100
9200
9300
9400
9500
9600
9700
9800
9900
10000
10100
10200
10300
10400
10500
10600
10700
10800
10900
11000
11100
11200
11300
11400
11500
11600
11700
11800
11900
12000
12100
12200
12300
12400
12500
12600
12700
12800
12900
13000
13100
13200
13300
13400
13500
13600
13700
13800
13900
14000
14100
14200
14300
14400
14500
14600
14700
14800
14900
15000
15100
15200
15300
15400
15500
15600
15700
15800
15900
16000
16100
16200
16300
16400
16500
16600
16700
16800
16900
17000
17100
17200
17300
17400
17500
17600
17700
17800
17900
18000
18100
18200
18300
18400
18

KeyboardInterrupt: 

In [ ]:
bins_1kb

In [6]:
# Use Specific Gene's closest regions to their start and end
gene = 'GATA1'

gene_chr = gene_hg19_bed.loc[gene].chr
gene_start = gene_hg19_bed.loc[gene].start
gene_end = gene_hg19_bed.loc[gene].end

print(gene_hg19_bed.loc[gene])
print(gene_start)
print(gene_end)
# print(bins_25kb[bins_25kb['seqname']==gene_chr].loc[96548])

print(bins_25kb.loc[(bins_25kb['seqname'] == gene_chr) & (bins_25kb['start'] <= gene_start)] & (bins_25kb['end'] >= gene_end))
# print(bins_25kb.loc[(bins_25kb['seqname'] == gene_chr) & (bins_25kb['start'] <= gene_start)].iloc[-1])
# print(bins_25kb.loc[(bins_25kb['seqname'] == gene_chr) & (bins_25kb['end'] >= gene_end)].iloc[0])

# region = bins_25kb.loc[(bins_25kb['seqname'] == gene_chr) & (bins_25kb['start'] <= gene_start)].iloc[-1]
region = bins_25kb.loc[(bins_25kb['seqname'] == gene_chr) & (bins_25kb['end'] >= gene_end)].iloc[0]

chr                                  chrX
start                            48644961
end                              48652715
4                                     200
strand                                  +
6                         ENST00000376670
7                                    4170
8                          protein-coding
descriptor         GATA binding protein 1
10                               48649516
11                               48652571
12                92,239,378,146,126,516,
13            0,4536,5289,5768,6617,7238,
Name: GATA1, dtype: object
48644961
48652715


KeyboardInterrupt: 

In [ ]:
### USING A RANDOM REGION ###
rand_row = random.randint(0, len(bins_25kb))
rand_row
region = bins_25kb.iloc[rand_row]
print(region)

# 25kb 

In [4]:
# Use Specific Gene's closest regions
gene = 'GATA1'

gene_chr = gene_hg19_bed.loc[gene].chr
gene_start = gene_hg19_bed.loc[gene].start
gene_end = gene_hg19_bed.loc[gene].end

print(gene_hg19_bed.loc[gene])
print(gene_start)
print(gene_end)
# print(bins_25kb[bins_25kb['seqname']==gene_chr].loc[96548])

print(bins_25kb.loc[(bins_25kb['seqname'] == gene_chr) & (bins_25kb['start'] <= gene_start)].iloc[-1])
print(bins_25kb.loc[(bins_25kb['seqname'] == gene_chr) & (bins_25kb['end'] >= gene_end)].iloc[0])

# region = bins_25kb.loc[(bins_25kb['seqname'] == gene_chr) & (bins_25kb['start'] <= gene_start)].iloc[-1]
region = bins_25kb.loc[(bins_25kb['seqname'] == gene_chr) & (bins_25kb['end'] >= gene_end)].iloc[0]

chr                                  chrX
start                            48644961
end                              48652715
4                                     200
strand                                  +
6                         ENST00000376670
7                                    4170
8                          protein-coding
descriptor         GATA binding protein 1
10                               48649516
11                               48652571
12                92,239,378,146,126,516,
13            0,4536,5289,5768,6617,7238,
Name: GATA1, dtype: object
48644961
48652715
seqname        chrX
start      48583800
end        48608800
?           7.49548
Name: 96547, dtype: object
seqname        chrX
start      48647800
end        48672800
?           7.49905
Name: 96548, dtype: object


In [56]:
### USING A RANDOM REGION ###
rand_row = random.randint(0, len(bins_25kb))
rand_row
region = bins_25kb.iloc[rand_row]
print(region)

seqname        chr8
start      65632200
end        65657200
?           5.15043
Name: 87637, dtype: object


In [7]:
region_index = region.name
region_chr = region.seqname
region_start = region.start
region_end = region.end

In [92]:
### SNP ###

snp_chr = snp_df.loc[snp_df['Chromosome'] == region_chr]
snp_in_region = snp_chr[(snp_chr['p1'] >= region_start) & (snp_chr['p2'] <= region_end)]

trait_codes_temp = trait_codes_df.set_index(trait_codes_df['phenotype'])

snp_in_region['description'] = snp_in_region['phenotype'].map(trait_codes_temp['description'])
snp_in_region.sort_values(by='pvalue')['description']

/tmp/ipykernel_147156/3853771718.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  snp_in_region['description'] = snp_in_region['phenotype'].map(trait_codes_temp['description'])


2566016                  Job coding: printer, master printer
2934073                             Job SOC coding: Printers
1739839    Treatment/medication code: isotard 25xl m/r ta...
8937313                   Mean platelet (thrombocyte) volume
1494527         Treatment/medication code: otomize ear spray
4017819        Diagnoses - main ICD10: C81 Hodgkin's disease
2442517    Job coding: personnel or industrial relations ...
2442518    Job coding: personnel or industrial relations ...
4241220    Diagnoses - main ICD10: G12 Spinal muscular at...
3954636                       Malignant neoplasm of pancreas
3861423                       Malignant neoplasm of pancreas
2223163    Job coding: financial services manager, charte...
2223162    Job coding: financial services manager, charte...
2705400    Job SOC coding: Financial managers and charter...
2705399    Job SOC coding: Financial managers and charter...
2374008    Job coding: medical radiographer, x-ray operat...
2814423                J

In [59]:
trait_codes_df

,phenotype,description,variable_type,source,n_non_missing,n_missing,n_controls,n_cases,PHESANT_transformation,notes
0,G54,Diagnoses - main ICD10: G54 Nerve root and ple...,categorical,icd10,361194,0,361051,143,NA,NA
1,C21,Diagnoses - main ICD10: C21 Malignant neoplasm...,categorical,icd10,361194,0,361055,139,NA,NA
2,F45,Diagnoses - main ICD10: F45 Somatoform disorders,categorical,icd10,361194,0,360953,241,NA,NA
3,I35,Diagnoses - main ICD10: I35 Nonrheumatic aorti...,categorical,icd10,361194,0,360293,901,NA,NA
4,D38,Diagnoses - main ICD10: D38 Neoplasm of uncert...,categorical,icd10,361194,0,361087,107,NA,NA
...,...,...,...,...,...,...,...,...,...,...
4354,100023_raw,Starch,continuous_raw,phesant,51453,309741,NA,NA,100023_0|| CONTINUOUS MAIN || CONTINUOUS || ||,"Starch. Estimated intake, based on food and be..."
4355,100024_raw,Calcium,continuous_raw,phesant,51453,309741,NA,NA,100024_0|| CONTINUOUS MAIN || CONTINUOUS || ||,"Calcium. Estimated intake, based on food and b..."
4356,100025_raw,Vitamin E,continuous_raw,phesant,51453,309741,NA,NA,100025_0|| CONTINUOUS MAIN || CONTINUOUS || ||,"Vitamin E. Estimated intake, based on food and..."
4357,is_female,"is_female, based on inferred genetic sex",categorical,covariate,361194,0,167020,194174,NA,NA


In [5]:
### TF ###
motif_counts_df = pd.read_pickle('../data/motif_counts_df_25kb.pkl')
motif_counts_df.iloc[:, 3:] = motif_counts_df.iloc[:, 3:].astype(int)

bin_cols = ['seqname', 'start', 'end', '?']
bins_25kb = pd.read_csv('../data/Altius_100k_epilogos_hg19_WM20210518.bed', delimiter='\t', names=bin_cols)
print(bins_25kb)

# motif_counts_chr = motif_counts_df[motif_counts_df['seqname'] == region_chr]
# motif_sub = motif_counts_chr.loc[(motif_counts_chr['start'] >= region_start) & (motif_counts_chr['end'] <= region_end)] 
# print(motif_sub)


total_tf_cluster_counts_expected = motif_counts_df.iloc[:, 3:].sum(axis=0)/len(bins_25kb)
# print(motif_sub.iloc[:, 3:])

# (np.log2(motif_sub.iloc[:, 3:]/total_tf_cluster_counts_expected)).T

motif_enrichment = (np.log2(motif_counts_df.iloc[:, 3:]/total_tf_cluster_counts_expected))
print(motif_enrichment)
print(motif_counts_df.iloc[:, :3])

motif_enrichment = pd.concat([motif_counts_df.iloc[:, :3], motif_enrichment], axis=1)
print(motif_enrichment)

motif_enrichment.to_pickle('../data/motif_log2_enrichment.pkl')

      seqname      start        end     ?
0        chr1       9000      34000  0.41
1        chr1      34000      59000  0.41
2        chr1      61000      86000  0.41
3        chr1      86000     111000  0.41
4        chr1     111000     136000  0.41
...       ...        ...        ...   ...
99991    chrX  155042800  155067800  0.41
99992    chrX  155067800  155092800  0.41
99993    chrX  155092800  155117800  0.41
99994    chrX  155117800  155142800  0.41
99995    chrX  155142800  155167800  0.41

[99996 rows x 4 columns]
            AHR      AIRE     AP1_1     AP1_2     ARI5A     ARI5B      BATF  \
0      0.940513  0.782845 -0.316858  0.755815 -0.981118 -0.449452 -0.516992   
1          -inf  0.782845  0.217478 -0.393048  0.018882 -0.449452  0.120438   
2     -0.644449  0.782845 -0.419952 -0.524293 -0.396155 -0.034414 -1.101954   
3      0.677479 -0.217155  0.410123  0.212673  0.018882  0.550548  0.120438   
4          -inf -1.217155 -0.087376  0.509654  0.826237  0.772941 -0.101954

In [63]:
### DHS ###
# Which DHSs are in the region
dhs_metadata_chr = dhs_df[dhs_df['seqname'] == region_chr]
dhs_metadata_region = dhs_metadata_chr.loc[(dhs_metadata_chr['start'] >= region_start) & (dhs_metadata_chr['end'] <= region_end)] 
# print(dhs_metadata_region)

# Component Enrichments in the region (obs/exp) # NOT LOG2 - empty values filled with 0
print(dhs_obs_over_exp.loc[region_index].iloc[3:])

# Average mean signal of DHSs
average_mean_signal = dhs_metadata_region['mean_signal'].mean()
print(average_mean_signal)

# Number of DHSs
print(len(dhs_metadata_region))

Cancer / epithelial        0.530514
Cardiac                         0.0
Digestive                  1.382296
Lymphoid                   1.427535
Musculoskeletal            0.461036
Myeloid / erythroid        1.071677
Neural                     1.300118
Organ devel. / renal       1.256728
Placental / trophoblast    2.641603
Primitive / embryonic        0.4788
Pulmonary devel.                0.0
Renal / cancer             3.469987
Stromal A                       0.0
Stromal B                       0.0
Tissue invariant           6.976318
Vascular / endothelial          0.0
Name: 87637, dtype: object
1.1422072457644274
44


In [19]:
### EPILOGOS ###
epilogos_chr = epilogos_states_df[epilogos_states_df['seqname'] == region_chr]
epilogos_in_region = epilogos_chr[(epilogos_chr['start'] >= region_start) & (epilogos_chr['end'] <= region_end)]
print(epilogos_in_region)


epilogos_region = epilogos_in_region.loc[:,list(range(1,19))].mean(axis=0)
# epilogos_region

epilogos_metadata_sub = epilogos_metadata[['one_index', 'short_name', 'long_name']]
epilogos_metadata_sub.set_index(epilogos_metadata_sub['one_index'], inplace=True)

epilogos_metadata_sub['score'] = epilogos_region
epilogos_metadata_sub

         seqname     start       end         1         2        3         4  \
14648395    chrX  48647800  48648000 -0.002432  0.012880  0.02569  0.022570   
14648396    chrX  48648000  48648200 -0.002416  0.005777  0.02170  0.017740   
14648397    chrX  48648200  48648400 -0.002432 -0.000957  0.02170  0.013210   
14648398    chrX  48648400  48648600 -0.000062  0.002838  0.01427  0.002109   
14648399    chrX  48648600  48648800 -0.002432  0.030310  0.00000 -0.000146   
...          ...       ...       ...       ...       ...      ...       ...   
14648515    chrX  48671800  48672000  0.000000  0.000000  0.00000  0.000000   
14648516    chrX  48672000  48672200  0.000000  0.000000  0.00000  0.000000   
14648517    chrX  48672200  48672400  0.000000  0.000000  0.00000  0.000000   
14648518    chrX  48672400  48672600  0.000000  0.000000  0.00000  0.000000   
14648519    chrX  48672600  48672800  0.000000  0.000000  0.00000  0.000000   

                 5        6         7  ...         

/tmp/ipykernel_90792/3882697906.py:13: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  epilogos_metadata_sub['score'] = epilogos_region


,one_index,short_name,long_name,score
one_index,,,,
1,1,TssA,Active TSS,4.754877e-01
2,2,TssFlnk,Flanking TSS,4.789110e-02
3,3,TssFlnkU,Flanking TSS Upstream,1.477751e-01
4,4,TssFlnkD,Flanking TSS Downstream,1.464048e-02
5,5,Tx,Strong transcription,2.432579e-02
6,6,TxWk,Weak transcription,6.841416e-01
7,7,EnhG1,Genic Enhancer 1,9.729920e-04
8,8,EnhG2,Genic Enhancer 2,2.963360e-04
9,9,EnhA1,Active Enhancer 1,2.648324e-02


In [129]:
epilogos_metadata

,zero_index,one_index,short_name,long_name,hex,rgba,color
0,0,1,TssA,Active TSS,#ff0000,"rgba(255,0,0,1)",Red
1,1,2,TssFlnk,Flanking TSS,#ff4500,"rgba(255,69,0,1)",Orange Red
2,2,3,TssFlnkU,Flanking TSS Upstream,#ff4500,"rgba(255,69,0,1)",Orange Red
3,3,4,TssFlnkD,Flanking TSS Downstream,#ff4500,"rgba(255,69,0,1)",Orange Red
4,4,5,Tx,Strong transcription,#008000,"rgba(0,128,0,1)",Green
5,5,6,TxWk,Weak transcription,#006400,"rgba(0,100,0,1)",DarkGreen
6,6,7,EnhG1,Genic Enhancer 1,#c2e105,"rgba(194,225,5,1)",GreenYellow
7,7,8,EnhG2,Genic Enhancer 2,#c2e105,"rgba(194,225,5,1)",GreenYellow
8,8,9,EnhA1,Active Enhancer 1,#ffc34d,"rgba(255,195,77,1)",Orange
9,9,10,EnhA2,Active Enhancer 2,#ffc34d,"rgba(255,195,77,1)",Orange


In [37]:
### GENE ###
gene_chr = gene_hg19_bed.loc[gene_hg19_bed['chr'] == region_chr]
genes_in_region = gene_chr[((gene_chr['start'] >= region_start) & (gene_chr['start'] <= region_end)) | ((gene_chr['end'] <= region_end) & (gene_chr['end'] >= region_start))]
print(genes_in_region)

print(region_start, region_end)

             chr     start       end    4 strand                6      7  \
gene_names                                                                 
HDAC6       chrX  48660401  48683386  212      +  ENST00000334136  14064   
GATA1       chrX  48644961  48652715  200      +  ENST00000376670   4170   

                         8              descriptor        10        11  \
gene_names                                                               
HDAC6       protein-coding   histone deacetylase 6  48661099  48683022   
GATA1       protein-coding  GATA binding protein 1  48649516  48652571   

                                                           12  \
gene_names                                                      
HDAC6       148,123,129,89,85,41,97,98,105,69,127,66,60,90...   
GATA1                                 92,239,378,146,126,516,   

                                                           13  
gene_names                                                     
HDAC6    

In [238]:
# Extra gene tester
gene2_start = gene_hg19_bed.loc['PPP1R3F'].start
gene2_end = gene_hg19_bed.loc['PPP1R3F'].end
dhs_metadata_chr = dhs_df[dhs_df['seqname'] == 'chrX']
dhs_metadata_region = dhs_metadata_chr.loc[(dhs_metadata_chr['start'] >= region_start) & (dhs_metadata_chr['end'] <= region_end)] 
print(dhs_metadata_region.component.sort_values(ascending=False))

identifier
X.380685    Vascular / endothelial
X.38053           Tissue invariant
X.380682                 Stromal B
X.380678            Renal / cancer
X.380691            Renal / cancer
X.38065      Primitive / embryonic
X.380662     Primitive / embryonic
X.38054      Primitive / embryonic
X.38059                     Neural
X.380675                    Neural
X.380695       Myeloid / erythroid
X.380688       Myeloid / erythroid
X.38064        Myeloid / erythroid
X.380668       Myeloid / erythroid
X.38056            Musculoskeletal
X.380698                  Lymphoid
X.380671                  Lymphoid
X.38062                   Lymphoid
X.38061                  Digestive
X.38058                  Digestive
X.38055                  Digestive
X.380665                 Digestive
Name: component, dtype: object


In [67]:
### STRING ###
protein_assoc_hg19 = pd.read_pickle('../data/hg19-protein-associations-detailed.pkl')
print(protein_assoc_hg19.shape)

protein_assoc_hg19.index = protein_assoc_hg19['protein1_pref']

columns = ['coexpression', 'experimental', 'database', 'textmining']
subscores = protein_assoc_hg19[columns]
print(subscores.shape)

subscores_600 = subscores.loc[subscores.max(axis=1) > 150]

(10538486, 11)
(10538486, 4)


In [68]:
genes_in_region_assoc = subscores_600.loc[genes_in_region.index]
subscores_count = pd.DataFrame()
for col in genes_in_region_assoc.columns:
    column = genes_in_region_assoc[col]
    subscores_count[col] = [len(column[column > 150])]
print(subscores_count.T)

                 0
coexpression    54
experimental   947
database       177
textmining    2140


In [66]:
protein_assoc_hg19.combined_score.min()

150.0

In [85]:
def create_region_vector():
    region_vector = []
    tf_arr = np.array(tf_vector(region_chr, region_start, region_end).iloc[0, 3:])
    dhs_arr = dhs_vector(region_chr, region_start, region_end)
    epilogos_arr = np.array(epilogos_vector(region_chr, region_start, region_end))
    # print(gene_vector(region_chr, region_start, region_end).tolist())
    gene_arr = gene_vector(region_chr, region_start, region_end).tolist()
    region_vector = np.concatenate((tf_arr, dhs_arr, epilogos_arr, gene_arr))
    return region_vector

322


In [84]:
def tf_vector(region_chr, region_start, region_end):
    motif_enrichment_chr = motif_enrichment[motif_enrichment['seqname'] == region_chr]
    motif_enrichment_region = motif_enrichment_chr.loc[(motif_enrichment_chr['start'] >= region_start) & (motif_enrichment_chr['end'] <= region_end)] 
    
    return motif_enrichment_region

def dhs_vector(region_chr, region_start, region_end):
    # Which DHSs are in the region
    dhs_metadata_chr = dhs_df[dhs_df['seqname'] == region_chr]
    dhs_metadata_region = dhs_metadata_chr.loc[(dhs_metadata_chr['start'] >= region_start) & (dhs_metadata_chr['end'] <= region_end)] 

#     # Component Enrichments in the region (obs/exp) # NOT LOG2 - empty values filled with 0
#     dhs_obs_over_exp_chr = dhs_obs_over_exp[dhs_obs_over_exp['seqname'] == region_chr]
#     dhs_obs_over_exp_region = dhs_obs_over_exp_chr.loc[(dhs_obs_over_exp_chr['start'] >= region_start) & (dhs_obs_over_exp_chr['end'] <= region_end)] 


    # Average mean signal of DHSs
    average_mean_signal = dhs_metadata_region['mean_signal'].mean()

    # Number of DHSs
    num_dhs = len(dhs_metadata_region)

    dhs_vector = [average_mean_signal, num_dhs]
    dhs_vector.extend(dhs_obs_over_exp_region.iloc[:, 3:].T.iloc[:, 0].tolist())
    
    del dhs_metadata_chr, dhs_metadata_region, dhs_obs_over_exp_chr, dhs_obs_over_exp_region
    
    return dhs_vector

def epilogos_vector(region_chr, region_start, region_end):
    epilogos_chr = epilogos_states_df[epilogos_states_df['seqname'] == region_chr]
    epilogos_in_region = epilogos_chr[(epilogos_chr['start'] >= region_start) & (epilogos_chr['end'] <= region_end)]

    epilogos_region = epilogos_in_region.loc[:,list(range(1,19))].mean(axis=0)
    return epilogos_region

def gene_vector(region_chr, region_start, region_end):
    gene_chr = gene_hg19_bed.loc[gene_hg19_bed['chr'] == region_chr]
    genes_in_region = gene_chr[((gene_chr['start'] >= region_start) & (gene_chr['start'] <= region_end)) | ((gene_chr['end'] <= region_end) & (gene_chr['end'] >= region_start))]

    protein_assoc_hg19 = pd.read_pickle('../data/hg19-protein-associations-detailed.pkl')
    protein_assoc_hg19.index = protein_assoc_hg19['protein1_pref']

    columns = ['coexpression', 'experimental', 'database', 'textmining']
    subscores = protein_assoc_hg19[columns]

    subscores_600 = subscores.loc[subscores.max(axis=1) > 150]

    genes_in_region_assoc = subscores_600.loc[genes_in_region.index]
    # Get mean of the subscores of gene associations
    return genes_in_region_assoc.mean(axis=0)

In [73]:
genes_in_region_assoc = subscores_600.loc[genes_in_region.index]
print(genes_in_region_assoc.mean(axis=0).tolist())

[31.261652542372882, 106.93538135593221, 44.38559322033898, 232.82909604519773]


In [89]:
# gene_chr = gene_hg19_bed.loc[gene_hg19_bed['chr'] == region_chr]
# genes_in_region = gene_chr[((gene_chr['start'] >= region_start) & (gene_chr['start'] <= region_end)) | ((gene_chr['end'] <= region_end) & (gene_chr['end'] >= region_start))]

protein_assoc_hg19 = pd.read_pickle('../data/hg19-protein-associations-detailed.pkl')
protein_assoc_hg19.index = protein_assoc_hg19['protein1_pref']

columns = ['coexpression', 'experimental', 'database', 'textmining']
subscores = protein_assoc_hg19[columns]

subscores_600 = subscores.loc[subscores.max(axis=1) > 150]

total = pd.DataFrame()

for gene in genes_in_region.index.tolist():
    if gene in subscores_600.index:
        genes_in_region_assoc = subscores_600.loc[genes_in_region.index]
        total = pd.concat([total, genes_in_region_assoc], axis=0)

# Get mean of the subscores of gene associations
print(total.mean(axis=0))

coexpression     31.261653
experimental    106.935381
database         44.385593
textmining      232.829096
dtype: float64


In [91]:
print(subscores_600.index)

Index(['ARF5', 'ARF5', 'ARF5', 'ARF5', 'ARF5', 'ARF5', 'ARF5', 'ARF5', 'ARF5',
       'ARF5',
       ...
       'PAK4', 'PAK4', 'PAK4', 'PAK4', 'PAK4', 'PAK4', 'PAK4', 'PAK4', 'PAK4',
       'PAK4'],
      dtype='object', name='protein1_pref', length=8352353)


In [95]:
gene_hg19_bed_STRING = gene_hg19_bed.loc[protein_assoc_hg19.index]
gene_hg19_bed_STRING

,chr,start,end,4,strand,6,7,8,descriptor,10,11,12,13
protein1_pref,,,,,,,,,,,,,
ARF5,chr7,127228398,127231759,6,+,ENST00000000233,658,protein-coding,ADP ribosylation factor 5,127228552,127231353,"221,81,110,72,126,493,","0,738,1140,1721,2618,2868,"
ARF5,chr7,127228398,127231759,6,+,ENST00000000233,658,protein-coding,ADP ribosylation factor 5,127228552,127231353,"221,81,110,72,126,493,","0,738,1140,1721,2618,2868,"
ARF5,chr7,127228398,127231759,6,+,ENST00000000233,658,protein-coding,ADP ribosylation factor 5,127228552,127231353,"221,81,110,72,126,493,","0,738,1140,1721,2618,2868,"
ARF5,chr7,127228398,127231759,6,+,ENST00000000233,658,protein-coding,ADP ribosylation factor 5,127228552,127231353,"221,81,110,72,126,493,","0,738,1140,1721,2618,2868,"
ARF5,chr7,127228398,127231759,6,+,ENST00000000233,658,protein-coding,ADP ribosylation factor 5,127228552,127231353,"221,81,110,72,126,493,","0,738,1140,1721,2618,2868,"
...,...,...,...,...,...,...,...,...,...,...,...,...,...
PAK4,chr19,39616473,39673456,92,+,ENST00000599386,16059,protein-coding,p21 (RAC1) activated kinase 4,39660193,39669219,"86,73,226,435,134,127,126,135,4393,","0,42396,43698,47742,49097,49478,50756,51841,52..."
PAK4,chr19,39616473,39673456,92,+,ENST00000599386,16059,protein-coding,p21 (RAC1) activated kinase 4,39660193,39669219,"86,73,226,435,134,127,126,135,4393,","0,42396,43698,47742,49097,49478,50756,51841,52..."
PAK4,chr19,39616473,39673456,92,+,ENST00000599386,16059,protein-coding,p21 (RAC1) activated kinase 4,39660193,39669219,"86,73,226,435,134,127,126,135,4393,","0,42396,43698,47742,49097,49478,50756,51841,52..."


In [96]:
## Get Cosine Similarity 
region_vectors_25kb_df = pd.read_pickle('../output/25kb_region_vectors.pkl')

for i in range(len(region_vectors_25kb_df)):
    

In [98]:
motif_enrichment = pd.read_pickle('../data/motif_log2_enrichment.pkl')
dhs_df = pd.read_pickle('../data/dhs_metadata.pkl')
epilogos_states_df = pd.read_pickle('../data/epilogos_states_df')

motif_enrichment_bed = BedTool.from_dataframe(motif_enrichment)
dhs_bed = BedTool.from_dataframe(dhs_df)
epilogos_states_bed = BedTool.from_dataframe(epilogos_states_df)

motif_enrichment_bed.saveas('../data/motif_enrichment_bed.bed')
dhs_bed.saveas('../data/dhs_bed.bed')
epilogos_states_bed.saveas('../data/epilogos_states_bed.bed')

<BedTool(../data/epilogos_states_bed.bed)>

In [99]:
motif_enrichment

,seqname,start,end,AHR,AIRE,AP1_1,AP1_2,ARI5A,ARI5B,BATF,...,ZNF652,ZNF667,ZNF680,ZNF708,ZNF713,ZNF768,ZNF784,ZNF85,ZSCAN3,ZSCAN4
0,chr1,9000,34000,0.940513,0.782845,-0.316858,0.755815,-0.981118,-0.449452,-0.516992,...,-0.332600,1.177511,0.602383,1.871040,-inf,1.339572,0.485744,-0.213453,1.370299,0.189086
1,chr1,34000,59000,-inf,0.782845,0.217478,-0.393048,0.018882,-0.449452,0.120438,...,-0.332600,-0.922025,-0.204972,-1.673281,-0.010709,0.252110,-0.099218,-0.097975,-0.025630,0.101624
2,chr1,61000,86000,-0.644449,0.782845,-0.419952,-0.524293,-0.396155,-0.034414,-1.101954,...,-0.332600,-0.729380,0.602383,0.063685,-0.010709,-0.162928,-0.514256,0.661016,-1.103632,-0.898376
3,chr1,86000,111000,0.677479,-0.217155,0.410123,0.212673,0.018882,0.550548,0.120438,...,0.667400,-0.559455,0.602383,0.063685,-inf,0.126579,-0.099218,-0.213453,0.118760,1.311077
4,chr1,111000,136000,-inf,-1.217155,-0.087376,0.509654,0.826237,0.772941,-0.101954,...,-0.917562,0.794182,0.495468,0.201188,0.574254,-0.332853,0.222710,-0.097975,0.770837,-1.091021
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
99991,chrX,155042800,155067800,0.355551,-0.632193,0.348723,0.331317,-0.396155,-0.449452,0.120438,...,0.889793,0.270620,-0.397617,0.201188,-1.010709,0.126579,0.900782,-0.628490,-0.186094,-0.576448
99992,chrX,155067800,155092800,1.162906,0.104773,-0.267949,-0.457179,1.018882,-0.034414,0.013523,...,0.082438,0.358083,0.116957,0.201188,0.574254,0.837072,-0.514256,-0.338984,0.584424,0.008514
99993,chrX,155092800,155117800,-0.059487,0.590200,-0.174839,0.170853,0.018882,-0.449452,-0.364989,...,0.667400,-0.269948,-0.397617,-0.258243,-0.010709,-1.010925,-1.099218,-0.338984,-1.273557,-0.197937
99994,chrX,155117800,155142800,0.355551,0.104773,-0.267949,-0.594682,0.018882,-0.034414,-0.879562,...,0.082438,0.592548,-0.397617,-0.450889,0.574254,-0.010925,-0.099218,-0.338984,-0.103632,0.493941


In [119]:
dhs_ms_num = pd.DataFrame()
for i in tqdm(range(len(bins_25kb))):
    region_chr = bins_25kb.iloc[i, 0]
    region_start = bins_25kb.iloc[i, 1]
    region_end = bins_25kb.iloc[i, 2]
#     print(region_chr, region_start, region_end)

    new_vector = dhs_vector(region_chr, region_start, region_end)
    dhs_ms_num[i] = new_vector
dhs_ms_num 

  0%|▏                                                                                                                                                                                                                  | 100/99996 [00:21<5:54:52,  4.69it/s]/tmp/ipykernel_10193/586572900.py:9: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead.  To get a de-fragmented frame, use `newframe = frame.copy()`
  dhs_ms_num[i] = new_vector
100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 99996/99996 [5:48:35<00:00,  4.78it/s]


,0,1,2,3,4,5,6,7,8,9,...,99986,99987,99988,99989,99990,99991,99992,99993,99994,99995
0,0.129388,0.176642,0.167473,0.148034,0.079265,0.203499,0.498372,0.176563,NaN,NaN,...,0.368483,0.202598,0.474190,0.445144,0.745700,0.616913,0.497687,0.320222,0.340393,0.360654
1,1.000000,2.000000,4.000000,6.000000,3.000000,1.000000,5.000000,7.000000,0.0,0.0,...,13.000000,13.000000,29.000000,19.000000,28.000000,30.000000,26.000000,20.000000,17.000000,13.000000
2,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.654274,0.0,0.0,...,0.654274,1.308548,1.308548,0.000000,1.308548,0.654274,1.308548,1.962823,0.654274,3.271371
3,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.0,...,0.000000,0.000000,1.043255,5.216275,1.043255,0.000000,2.086510,1.043255,0.000000,1.043255
4,0.000000,0.000000,0.857702,0.857702,0.857702,0.000000,0.000000,0.000000,0.0,0.0,...,0.000000,0.000000,0.000000,0.000000,1.715403,0.857702,1.715403,0.857702,0.857702,0.000000
5,0.000000,0.000000,0.000000,0.441178,0.441178,0.000000,0.882355,0.000000,0.0,0.0,...,0.000000,0.882355,3.529421,0.441178,0.882355,0.882355,0.882355,0.441178,0.000000,0.882355
6,0.000000,0.000000,0.000000,0.569736,0.000000,0.000000,0.000000,0.000000,0.0,0.0,...,0.000000,0.000000,0.569736,2.278942,0.569736,1.139471,0.569736,0.569736,0.569736,0.569736
7,0.000000,0.000000,0.000000,0.000000,0.000000,0.663350,0.000000,0.663350,0.0,0.0,...,1.326700,0.663350,0.663350,1.326700,1.990049,1.990049,2.653399,0.663350,2.653399,0.000000
8,0.000000,0.267417,0.000000,0.267417,0.000000,0.000000,0.000000,0.534834,0.0,0.0,...,1.069668,1.069668,0.534834,0.534834,0.267417,0.802251,0.267417,0.267417,0.267417,0.267417
9,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.0,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.776777,1.553553,0.000000,0.000000,0.776777


In [123]:
dhs_ms_num.T.to_pickle('../data/dhs_vector_25kb.pkl')
dhs_ms_num.T.to_pickle('../../../tmp/data/dhs_vector_25kb.pkl')

In [ ]:
epilogos_states_df

In [100]:
bins_25kb

,seqname,start,end,?
0,chr1,9000,34000,0.41
1,chr1,34000,59000,0.41
2,chr1,61000,86000,0.41
3,chr1,86000,111000,0.41
4,chr1,111000,136000,0.41
...,...,...,...,...
99991,chrX,155042800,155067800,0.41
99992,chrX,155067800,155092800,0.41
99993,chrX,155092800,155117800,0.41
99994,chrX,155117800,155142800,0.41


In [112]:
bins_25kb_bed = BedTool.from_dataframe(bins_25kb)
bins_25kb_bed

<BedTool(/tmp/pybedtools.x7wxn8ne.tmp)>

In [115]:
dhs_overlap = bins_25kb_bed.intersect(dhs_bed, u=True)
print(dhs_overlap.to_dataframe())

      chrom      start        end  name
0      chr1       9000      34000  0.41
1      chr1      34000      59000  0.41
2      chr1      61000      86000  0.41
3      chr1      86000     111000  0.41
4      chr1     111000     136000  0.41
...     ...        ...        ...   ...
93128  chrX  155042800  155067800  0.41
93129  chrX  155067800  155092800  0.41
93130  chrX  155092800  155117800  0.41
93131  chrX  155117800  155142800  0.41
93132  chrX  155142800  155167800  0.41

[93133 rows x 4 columns]


In [117]:
bins_25kb

,seqname,start,end,?
0,chr1,9000,34000,0.41
1,chr1,34000,59000,0.41
2,chr1,61000,86000,0.41
3,chr1,86000,111000,0.41
4,chr1,111000,136000,0.41
...,...,...,...,...
99991,chrX,155042800,155067800,0.41
99992,chrX,155067800,155092800,0.41
99993,chrX,155092800,155117800,0.41
99994,chrX,155117800,155142800,0.41


In [120]:
bins_25kb_bed

<BedTool(/tmp/pybedtools.x7wxn8ne.tmp)>